# FYP: Robust Deep Image Steganography Against Downscaling
Colab starter notebook. Run cells top to bottom.

**First: Runtime -> Change runtime type -> T4 GPU -> Save**

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU - go enable the GPU runtime!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone your repo
Push the `fyp_stego/` folder (this notebook's sibling files) to a GitHub repo first, then clone it here.
If you haven't made the repo yet, skip this cell and use the next one to upload files manually instead.

In [ ]:
# Option A: clone from GitHub (replace with your repo URL)
!git clone https://github.com/YOUR_USERNAME/fyp-stego.git
%cd fyp-stego

In [ ]:
!pip install -q pytorch-msssim scikit-image opencv-python-headless pyyaml tqdm pandas

## Download DIV2K straight into Drive (do this once; it persists across sessions)

In [ ]:
import os
DATA_DIR = '/content/drive/MyDrive/fyp_data'
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(f'{DATA_DIR}/DIV2K_train_HR'):
    !wget -q http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip -P {DATA_DIR}
    !unzip -q {DATA_DIR}/DIV2K_train_HR.zip -d {DATA_DIR}
if not os.path.exists(f'{DATA_DIR}/DIV2K_valid_HR'):
    !wget -q http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip -P {DATA_DIR}
    !unzip -q {DATA_DIR}/DIV2K_valid_HR.zip -d {DATA_DIR}

print('train:', len(os.listdir(f'{DATA_DIR}/DIV2K_train_HR')))
print('valid:', len(os.listdir(f'{DATA_DIR}/DIV2K_valid_HR')))

## Point config at Drive paths, set checkpoint dir to Drive too (survives disconnects)

In [ ]:
import yaml
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['train_dir'] = f'{DATA_DIR}/DIV2K_train_HR'
cfg['data']['val_dir'] = f'{DATA_DIR}/DIV2K_valid_HR'
cfg['train']['checkpoint_dir'] = '/content/drive/MyDrive/fyp_checkpoints'

with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print(cfg)

## Quick sanity check (shapes, no training)

In [ ]:
!python models.py

## Train
Runs for up to `train.epochs` (default 60), checkpointing every epoch to Drive.
If the session disconnects, re-run this cell with `--resume` pointing at `last.pt`.

In [ ]:
!python train.py --config configs/config.yaml

In [ ]:
# To resume after a disconnect, uncomment:
# !python train.py --config configs/config.yaml --resume /content/drive/MyDrive/fyp_checkpoints/last.pt

## Full evaluation (5 algorithms x 4 sizes x 100 images, CNN vs LSB)

In [ ]:
!python eval.py --config configs/config.yaml --checkpoint /content/drive/MyDrive/fyp_checkpoints/best.pt --out-dir /content/drive/MyDrive/fyp_results

In [ ]:
import pandas as pd
summary = pd.read_csv('/content/drive/MyDrive/fyp_results/eval_summary.csv')
summary